In [ ]:
"""
JAX + NumPyro vectorized Gibbs sampler for hierarchical Dirichlet process-like model
Converted and vectorized from a PyTorch+Pyro implementation.

Notes:
- Uses JAX (jax.numpy as jnp) for array ops, jit, vmap.
- Uses numpyro.distributions for log_prob helpers and some sampling.
- Uses Gumbel-argmax for categorical sampling (fast, vectorized).
- The implementation focuses on clarity and vectorization for per-doc operations.

Functions:
- mix_weights_sticks: convert stick-breaking beta samples -> mixture weights
- sample_beta_rng: sample Beta using jax.random via Gamma construction
- sample_dirichlet_rng: sample Dirichlet via Gamma
- initialize_state: initialize global and per-structure variables
- gibbs_step_for_doc_jax: vectorized single-doc Gibbs step (jit)
- run_gibbs: driver that runs iterations and returns final state

This is a large, self-contained file. Adjust shapes and indexing to match your `struct_upbd`.
"""

from functools import partial
import jax
import jax.numpy as jnp
import numpy as onp
from jax import random, vmap, jit
import numpyro.distributions as dist

# -------------------- helpers --------------------

def mix_weights_sticks(beta):
    """Convert stick-breaking Beta samples (K,) into K mixture weights that sum to 1.
    If beta is shape (..., K) returns same leading dims with K weights.
    Implementation follows standard stick-breaking: w_k = beta_k * prod_{j<k} (1 - beta_j)
    """
    # beta shape (..., K)
    one_m_beta = 1.0 - beta
    # compute cumulative product along last axis of previous (1-beta)
    # use exclusive cumprod: for k, prod_{j<k} (1-beta_j)
    exclusive_cumprod = jnp.concatenate([jnp.ones_like(beta[..., :1]), jnp.cumprod(one_m_beta[..., :-1], axis=-1)], axis=-1)
    weights = beta * exclusive_cumprod
    return weights


def suffix_sum(x):
    # sum over suffix along last axis (like reverse cumsum)
    return jnp.flip(jnp.cumsum(jnp.flip(x, axis=-1), axis=-1), axis=-1)


def gumbel_keys_sample(key, logits, axis=-1):
    # logits: arbitrary shape; sample argmax along axis via gumbel
    g = random.gumbel(key, logits.shape)
    samples = jnp.argmax(logits + g, axis=axis)
    return samples


def beta_sample_from_alpha_beta(key, a, b, shape=()):
    # sample Beta(a,b) via Gamma sampling: x ~ Gamma(a,1), y ~ Gamma(b,1) => x/(x+y) ~ Beta(a,b)
    k1 = random.split(key, 2)
    x = random.gamma(k1[0], a, shape=shape)
    y = random.gamma(k1[1], b, shape=shape)
    return x / (x + y)


def dirichlet_sample_from_alpha(key, alpha):
    # alpha: (..., K)
    g = random.gamma(key, alpha)
    return g / jnp.sum(g, axis=-1, keepdims=True)

# small stable logsumexp

def logsumexp(x, axis=-1, keepdims=False):
    m = jnp.max(x, axis=axis, keepdims=True)
    s = jnp.log(jnp.sum(jnp.exp(x - m), axis=axis, keepdims=True)) + m
    if not keepdims:
        s = jnp.squeeze(s, axis=axis)
    return s

# -------------------- model initialization --------------------

def initialize_state(rng_key, struct_upbd, vocab_size, N, M):
    """Initialize parameters and state. Returns dict with arrays on device.
    struct_upbd: dict with keys like 'G0','G1','G2' giving sizes.
    N: num docs, M: words per doc (if varying you can adapt)
    """
    keys = random.split(rng_key, 32)
    k = 0

    # simple priors
    G0 = struct_upbd['G0']
    G1 = struct_upbd.get('G1', 1)
    G2 = struct_upbd.get('G2', 1)

    state = {}

    # global gamma for Dirichlet generation components
    state['gamma'] = jnp.abs(random.normal(keys[k], (1,))) + 0.1; k += 1

    # NIG prior params
    state['nig_mu'] = jnp.zeros((G0,))
    state['nig_kappa'] = jnp.ones((G0,)) * 1.0
    state['nig_alpha'] = jnp.ones((G0,)) * 1.0
    state['nig_beta'] = jnp.ones((G0,)) * 1.0

    # alpha/eta for hierarchical Beta constructions
    # For simplicity we assume shapes: alpha0: (G1, G0) and alpha1: (G2, G1, G0), etc.
    # adapt as necessary to match your original shapes
    state['alpha0'] = jnp.abs(random.normal(keys[k], (G1, G0))) + 0.5; k += 1
    state['eta0'] = jnp.abs(random.normal(keys[k], (G1,))) + 0.5; k += 1
    state['alpha1'] = jnp.abs(random.normal(keys[k], (G2, G1, G0))) + 0.5; k += 1
    state['eta1'] = jnp.abs(random.normal(keys[k], (G2, G1))) + 0.5; k += 1

    # initialize stick-breaking Beta samples for structure (random)
    # B0: (G0,) for top-level
    state['B0'] = random.beta(keys[k], 1.0, 1.0, shape=(G0,)); k += 1
    state['G0_weights'] = mix_weights_sticks(state['B0'])[:-1]  # size G0

    # deeper levels B1: shape (G1, G0?) We will store as B1[s,c,k] where k indexes sticks across topics
    # For simplicity choose stick length = G0
    state['B1'] = random.beta(keys[k], 1.0, 1.0, shape=(G1, G0)); k += 1
    state['G1_weights'] = mix_weights_sticks(state['B1'])  # returns (G1, G0)

    state['B2'] = random.beta(keys[k], 1.0, 1.0, shape=(G2, G1, G0)); k += 1
    state['G2_weights'] = mix_weights_sticks(state['B2'])

    # cluster-level Betas L0 and L1 (cluster mixing)
    state['L0'] = random.beta(keys[k], 1.0, 1.0, shape=(G1,)); k += 1
    state['L1'] = random.beta(keys[k], 1.0, 1.0, shape=(G2, G1)); k += 1
    state['L0_w'] = mix_weights_sticks(state['L0'].reshape(1, -1))[0]
    state['L1_w'] = mix_weights_sticks(state['L1'])

    # mixture components: generation (G0, V)
    alpha_gen = state['gamma'] * jnp.ones((G0, vocab_size))
    state['gen'] = dirichlet_sample_from_alpha(keys[k], alpha_gen); k += 1

    # regression mixture (Normal with NIG prior) - initialize sigma and mu
    state['regression_sigma'] = jnp.ones((G0,))
    state['regression_mu'] = random.normal(keys[k], (G0,)); k += 1

    # data-related structures
    # For this demo, initialize random observed words (indices) and labels
    state['words_obs'] = random.randint(keys[k], (N, M), minval=0, maxval=vocab_size); k += 1
    # regression labels
    state['labels'] = random.normal(keys[k], (N,)); k += 1

    # topic assignments z_gen (N, M) and z_reg (N,)
    state['z_gen'] = random.randint(keys[k], (N, M), minval=0, maxval=G0); k += 1
    state['z_reg'] = random.randint(keys[k], (N,), minval=0, maxval=G0); k += 1

    # category assignments (N, 2) for (row, col)
    state['category_assignments'] = random.randint(keys[k], (N, 2), minval=0, maxval=jnp.maximum(G1, G2)); k += 1

    # doc-level Beta/weights (B and G for each doc)
    # Choose stick length = G0
    state['doc_B'] = random.beta(keys[k], 1.0, 1.0, shape=(N, G0)); k += 1
    state['doc_G'] = mix_weights_sticks(state['doc_B'])[..., :-1]

    # placeholders for other structures used in code
    state['struct_weights'] = {
        'B0': (jnp.ones((G0,)), jnp.ones((G0,))),
        'G0': state['G0_weights'],
        'B1': (jnp.ones((G1, G0)), jnp.ones((G1, G0))),
        'G1': state['G1_weights'],
        'B2': (jnp.ones((G2, G1, G0)), jnp.ones((G2, G1, G0))),
        'G2': state['G2_weights']
    }

    state['cluster_weights'] = {'L0': state['L0_w'], 'L1': state['L1_w']}
    state['struct_samples'] = {'B0': state['B0'], 'B1': state['B1'], 'B2': state['B2']}
    state['mixture_components'] = {'generation': state['gen'], 'regression_sigma': state['regression_sigma'], 'regression_mu': state['regression_mu']}
    state['struct_params'] = {'alpha0': state['alpha0'], 'alpha1': state['alpha1'], 'eta0': state['eta0'], 'eta1': state['eta1'], 'gamma': state['gamma'], 'nig_mu': state['nig_mu'], 'nig_kappa': state['nig_kappa'], 'nig_alpha': state['nig_alpha'], 'nig_beta': state['nig_beta']}

    return state

# -------------------- per-doc Gibbs step (vectorized) --------------------

@partial(jit, static_argnums=(2,3,4))
def gibbs_step_for_doc_jax(rng_key, doc_idx, struct_upbd, M, vocab_size, state):
    """Per-document Gibbs step. This function is jitted and expects state to be JAX arrays.
    rng_key: PRNGKey
    doc_idx: int
    M: words per doc
    vocab_size: V
    state: dict with current parameters and assignments
    Returns updated partial state for that doc (z_gen row, z_reg, category assignment, doc_B, doc_G)
    """
    G0 = struct_upbd['G0']
    G1 = struct_upbd.get('G1', 1)
    G2 = struct_upbd.get('G2', 1)

    # grab local data
    words = state['words_obs'][doc_idx]         # (M,)
    reg_val = state['labels'][doc_idx]          # scalar
    doc_B = state['doc_B'][doc_idx]             # (G0,)
    doc_G = state['doc_G'][doc_idx]             # (G0,)

    mixture = state['mixture_components']['generation']  # (G0, V)
    reg_mu = state['mixture_components']['regression_mu']
    reg_sigma = state['mixture_components']['regression_sigma']

    # RNG split for different parts
    k1, k2, k3, k4 = random.split(rng_key, 4)

    # --- 1) WORD TOPIC GIBBS (vectorized across M) ---
    # compute per-topic log-prob for each word
    log_p = jnp.log(mixture + 1e-20)  # (G0, V)
    # gather per-word per-topic: (G0, M)
    topic_logp_words = log_p[:, words]  # uses advanced indexing
    log_doc_prior = jnp.log(doc_G + 1e-20).reshape(G0, 1)
    log_post = topic_logp_words + log_doc_prior  # (G0, M)

    # sample topics for M words via gumbel-argmax across topic dim
    new_topics = gumbel_keys_sample(k1, log_post, axis=0)  # (M,)

    # --- 2) REGRESSION COMPONENT GIBBS ---
    # compute log-likelihood under each topic
    var = reg_sigma ** 2
    log_const = -0.5 * (jnp.log(2 * jnp.pi) + jnp.log(var + 1e-20))
    log_like = log_const - 0.5 * ((reg_val - reg_mu) ** 2) / (var + 1e-20)  # (G0,)
    log_post_reg = log_like + jnp.log(doc_G + 1e-20)
    new_z_reg = gumbel_keys_sample(k2, log_post_reg, axis=0)  # scalar

    # --- 3) STRUCTURAL COMPONENT GIBBS (s,c joint) ---
    # compute cluster log probs
    L0 = state['cluster_weights']['L0']  # (G1,)
    L1 = state['cluster_weights']['L1']  # (G2, G1)
    log_cluster = jnp.log(L1 + 1e-20) + jnp.log(L0 + 1e-20)[None, :]  # (G2, G1)

    # struct log prob: Beta log_probs on struct_samples
    # assume struct_samples['B1'] shape (G1, G0) and ['B2'] (G2, G1, G0) etc.
    B1_samples = state['struct_samples']['B1']  # (G1, G0)
    B0_a = state['struct_weights']['B0'][0]
    B0_b = state['struct_weights']['B0'][1]
    # Beta logpdf for B0 samples sum across topic-axis if needed
    logpdf_B0 = dist.Beta(B0_a, B0_b).log_prob(state['struct_samples']['B0']).sum(axis=-1)  # (G1,) or (G0,)

    # For B1 and others, create a (G2, G1) matrix of sums
    # Here we assume struct_samples['B1'] shaped (G1, G0) and struct_samples['B2'] shaped (G2, G1, G0)
    # Compute logpdf of B2 under B2 params (we'll fallback to simple sum) --- adapt shapes to your model
    try:
        B2_samples = state['struct_samples']['B2']  # (G2, G1, G0)
        B1_a = state['struct_weights']['B1'][0]
        B1_b = state['struct_weights']['B1'][1]
        # logpdf_B1 per (G1, G0) then sum over last axis to get (G1,)
        logpdf_B1_per = dist.Beta(B1_a, B1_b).log_prob(B1_samples).sum(axis=-1)  # (G1,)
        # For B2, compute per (G2,G1) by summing last axis
        B2_a = state['struct_weights']['B2'][0]
        B2_b = state['struct_weights']['B2'][1]
        logpdf_B2 = dist.Beta(B2_a, B2_b).log_prob(B2_samples).sum(axis=-1)  # (G2, G1)
        # combine
        # log_struct_prob[c,s] = logpdf_B2[c,s] + logpdf_B1_per[s]
        log_struct = logpdf_B2 + logpdf_B1_per[None, :]
    except Exception:
        # fallback simple zero
        log_struct = jnp.zeros_like(log_cluster)

    # doc-level Beta loglik (doc_B under BP)
    doc_alpha = state['doc_BP_alpha'][doc_idx] if 'doc_BP_alpha' in state else jnp.ones_like(doc_B)
    doc_beta  = state['doc_BP_beta'][doc_idx] if 'doc_BP_beta' in state else jnp.ones_like(doc_B)
    # numeric: sum Beta.log_prob over components
    log_doc_beta = dist.Beta(doc_alpha, doc_beta).log_prob(doc_B).sum()

    # word likelihoods and regression term (scalars independent of s,c in original code)
    # compute log_word_probs_total
    topic_idx = new_topics  # (M,)
    # word_log_probs: for each m, log_p[topic_m, word_m]
    word_log_probs = log_p[topic_idx, words]
    log_topic_prior_for_words = jnp.log(doc_G[topic_idx] + 1e-20)
    log_word_probs_total = jnp.sum(word_log_probs + log_topic_prior_for_words)

    reg_topic = new_z_reg
    log_reg_topic_prior = jnp.log(doc_G[reg_topic] + 1e-20)
    log_reg_prob_scalar = log_like[reg_topic]

    total_log = log_cluster + log_struct + log_doc_beta + log_word_probs_total + (log_reg_topic_prior + log_reg_prob_scalar)

    # normalize and sample joint (c,s)
    total_log = total_log - jnp.max(total_log)
    total_prob = jnp.exp(total_log)
    total_prob = total_prob / jnp.sum(total_prob)

    flat_logits = jnp.log(total_prob.reshape(-1) + 1e-20)
    sampled_flat = gumbel_keys_sample(k3, flat_logits, axis=0)
    c_idx = sampled_flat // total_prob.shape[1]
    s_idx = sampled_flat % total_prob.shape[1]

    # update document-level Beta and weights as in original
    # recompute data summary
    unique, counts = jnp.unique(new_topics, return_counts=True)
    data_bias = jnp.zeros((G0,))
    data_bias = data_bias.at[unique].set(counts.astype(jnp.float32))
    data_bias = data_bias.at[new_z_reg].add(1.0)
    alpha_bias = data_bias
    beta_bias = suffix_sum(data_bias)

    weights_prior = state['struct_weights'][f'G{len(struct_upbd)-1}'][c_idx, s_idx]
    concentrate = state['struct_params'][f'alpha{len(struct_upbd)-1}'][s_idx, c_idx]
    param_alpha = concentrate * weights_prior
    param_beta = concentrate * (1 - weights_prior.cumsum(-1))

    # sample new doc_B via Beta (componentwise)
    new_doc_B = beta_sample_from_alpha_beta(k4, param_alpha + alpha_bias, param_beta + beta_bias, shape=param_alpha.shape)
    new_doc_G = mix_weights_sticks(new_doc_B)[:-1]

    # return updates as a small dict
    out = {
        'z_gen_row': new_topics,
        'z_reg': new_z_reg,
        'category': jnp.stack([s_idx, c_idx]),
        'doc_B': new_doc_B,
        'doc_G': new_doc_G
    }
    return out

# -------------------- top-level Gibbs driver --------------------

def run_gibbs(rng_key, struct_upbd, vocab_size, N=50, M=200, iterations=500):
    # initialize
    state = initialize_state(rng_key, struct_upbd, vocab_size, N, M)

    # convert to jax arrays (they already are) and create doc-level BP if needed
    # placeholders for doc_BP
    state['doc_BP_alpha'] = jnp.ones_like(state['doc_B'])
    state['doc_BP_beta']  = jnp.ones_like(state['doc_B'])

    # prepare rng keys for docs per iteration
    key = rng_key
    for it in range(iterations):
        key, sub = random.split(key)
        keys_docs = random.split(sub, N)

        # vectorize gibbs_step_for_doc_jax over docs using vmap (but function expects scalar doc idx), so use vmap over indices
        vmapped = vmap(lambda k, idx: gibbs_step_for_doc_jax(k, idx, struct_upbd, M, vocab_size, state), in_axes=(0, 0))
        doc_indices = jnp.arange(N)
        res = vmapped(keys_docs, doc_indices)

        # write back updates to state
        state['z_gen'] = state['z_gen'].at[jnp.arange(N)].set(res['z_gen_row'])
        state['z_reg'] = res['z_reg']
        state['category_assignments'] = res['category'].T  # (N,2)
        state['doc_B'] = res['doc_B']
        state['doc_G'] = res['doc_G']

        # update global mixture components (generation) from counts
        # compute counts per topic per vocab
        # For speed, compute histogram: shape (G0, V)
        G0 = struct_upbd['G0']
        V = vocab_size
        # flatten per-topic counts
        z_flat = state['z_gen'].reshape(-1)
        words_flat = state['words_obs'].reshape(-1)
        # compute topic-word co-occurrence using one-hot gather (vectorized)
        counts = jnp.zeros((G0, V))
        counts = counts.at[z_flat, words_flat].add(1.0)

        gamma = state['struct_params']['gamma'] * jnp.ones((G0, V)) + counts
        # sample Dirichlet per topic
        key, sub = random.split(key)
        keys = random.split(sub, G0)
        gen = vmap(lambda k, a: dirichlet_sample_from_alpha(k, a))(keys, gamma)
        state['mixture_components']['generation'] = gen

        # update regression NIG posterior and sample regression mu and sigma per topic
        # compute per-topic sums and counts and unnormalized variance (we'll do simple unbiased estimates)
        labels = state['labels']
        z_reg = state['z_reg']
        # compute counts, sums, means, unnorm_vars
        counts_reg = jnp.zeros((G0,)).at[z_reg].add(1.0)
        sums = jnp.zeros((G0,)).at[z_reg].add(labels)
        means = jnp.where(counts_reg > 0, sums / (counts_reg + 1e-20), 0.0)
        # unnorm_vars approximate: sum (y - mean)^2 per topic
        # compute by mapping over topics
        def topic_var(t):
            mask = (z_reg == t)
            vals = labels * mask
            cnt = jnp.sum(mask)
            m = jnp.where(cnt > 0, jnp.sum(vals) / (cnt + 1e-20), 0.0)
            # unnorm var = sum (y - m)^2
            s = jnp.sum(((labels - m) * mask) ** 2)
            return s
        unnorm_vars = jnp.array([topic_var(t) for t in range(G0)])

        # posterior params
        nig_kappa = counts_reg + state['struct_params']['nig_kappa']
        nig_mu = (sums + state['struct_params']['nig_kappa'] * state['struct_params']['nig_mu']) / nig_kappa
        nig_alpha = counts_reg / 2.0 + state['struct_params']['nig_alpha']
        nig_beta = state['struct_params']['nig_beta'] + 0.5 * unnorm_vars + 0.5 * ((means - state['struct_params']['nig_mu']) ** 2) * counts_reg * state['struct_params']['nig_kappa'] / (nig_kappa + 1e-20)

        # sample regression sigma (InverseGamma) and mu
        key, sub = random.split(key)
        keys1 = random.split(sub, G0)
        def sample_inv_gamma(k, a, b):
            # numpyro InverseGamma uses concentration/scale -- use numpyro dist to sample
            return dist.InverseGamma(a, b).sample(k)
        reg_sigma = vmap(sample_inv_gamma)(keys1, nig_alpha, nig_beta)
        reg_mu = random.normal(key, (G0,)) * jnp.sqrt(reg_sigma / nig_kappa) + nig_mu
        state['mixture_components']['regression_sigma'] = reg_sigma
        state['mixture_components']['regression_mu'] = reg_mu

        # Optionally update struct_weights/Betas using analytic posteriors like earlier code (omitted for brevity)

    return state

# -------------------- example execution --------------------
if __name__ == '__main__':
    key = random.PRNGKey(0)
    struct_upbd = {'G0': 10, 'G1': 2, 'G2': 3}
    final_state = run_gibbs(key, struct_upbd, vocab_size=200, N=50, M=100, iterations=100)
    print('done')


In [ ]:
# JAX / NumPyro replacements & optimized helpers
from functools import partial
import jax
import jax.numpy as jnp
from jax import random, vmap, jit, lax
import numpy as onp
import numpyro.distributions as dist

# -------------------------
# generalized_einsum
# -------------------------
# A general, efficient einsum-like function for the specific pattern you used:
#   A: (n, a1, a2, ..., aL)
#   B: (aL, aL-1, ..., a1, k)
# returns: (n, k) performing contraction over the reversed a-dim order.
@jit
def generalized_einsum(tensor_a: jnp.ndarray, tensor_b: jnp.ndarray) -> jnp.ndarray:
    """
    Equivalent to PyTorch einsum pattern "n i j k l, l k j i z -> n z"
    but implemented more robustly using jnp.transpose and jnp.tensordot.
    """
    # shapes
    shape_a = tensor_a.shape
    shape_b = tensor_b.shape
    if len(shape_a) < 2:
        raise ValueError("tensor_a must have at least 2 dimensions (n, ...)")
    if len(shape_b) < 2:
        raise ValueError("tensor_b must have at least 2 dimensions (..., k)")

    n = shape_a[0]
    dims_a = shape_a[1:]                     # (a1, a2, ..., aL)
    dims_b = shape_b[:-1]                   # (aL, aL-1, ..., a1)
    k = shape_b[-1]

    if tuple(dims_a) != tuple(dims_b[::-1]):
        raise ValueError("B's leading dims (excluding last) must be reverse of A's trailing dims")

    # reorder B's leading dims to match A's order: B has (aL, a(L-1), ..., a1, k)
    # we want B_reordered of shape (a1, a2, ..., aL, k)
    perm = list(range(len(dims_b)-1, -1, -1)) + [len(dims_b)]  # reverse indices then last index (k)
    # But perm is indexes relative to original B (0..L, L is k). Example L=4 -> [3,2,1,0,4]
    B_reordered = jnp.transpose(tensor_b, axes=perm)

    # Now do tensordot over all dims a1..aL
    # tensordot axes: A axes 1..L with B_reordered axes 0..L-1
    result = jnp.tensordot(tensor_a, B_reordered, axes=(list(range(1, 1 + len(dims_a))), list(range(len(dims_a)))))
    # result shape: (n, k)
    return result

# -------------------------
# mix_weights (stick-breaking -> mixture weights)
# -------------------------
@jit
def mix_weights(beta: jnp.ndarray, eps: float = 1e-12) -> jnp.ndarray:
    """
    beta: (..., K) stick-breaking betas in (0,1)
    returns weights (..., K+1) normalized (last weight is the leftover)
    Implementation: w_k = beta_k * prod_{j < k} (1 - beta_j); last weight = prod_j (1 - beta_j)
    """
    one_m_beta = 1.0 - beta
    # exclusive cumprod of (1-beta): prod_{j<k} (1-beta_j)
    # compute cumprod of one_m_beta, then shift right and prepend 1
    cumprod = jnp.cumprod(one_m_beta, axis=-1)
    exclusive = jnp.concatenate([jnp.ones_like(cumprod[..., :1]), cumprod[..., :-1]], axis=-1)
    primary = beta * exclusive  # (..., K)
    last = jnp.prod(one_m_beta, axis=-1, keepdims=True)  # (..., 1)
    weights = jnp.concatenate([primary, last], axis=-1)
    # numerical guard and renormalize along last axis
    weights = jnp.clip(weights, a_min=eps)
    weights = weights / jnp.sum(weights, axis=-1, keepdims=True)
    return weights

# -------------------------
# stable log-likelihood computation (vectorized)
# -------------------------
@jit
def compute_log_likelihood(state):
    """
    state is expected to hold:
      - state['words']['z_gen']: (N, M) int32 indices of topics
      - state['words']['obs']: (N, M, V) one-hot OR (N, M) word indices.
        Prefer word indices in JAX pipeline. We'll accept either.
      - state['mixture_components']['generation']: (G0, V) topic-word probabilities
      - state['mixture_components']['regression_mu']: (G0,)
      - state['mixture_components']['regression_sigma']: (G0,) std or variance? We'll assume std.
      - state['words']['z_reg']: (N,)
      - state['words']['reg']: (N,)
    Returns scalar log-likelihood (float32).
    """
    z_gen = state['words']['z_gen']        # (N, M)
    z_reg = state['words']['z_reg']        # (N,)
    reg_y = state['words']['reg']          # (N,)
    gen = state['mixture_components']['generation']  # (G0, V)
    mu = state['mixture_components']['regression_mu']    # (G0,)
    sigma = state['mixture_components']['regression_sigma']  # (G0,) std

    N, M = z_gen.shape
    G0, V = gen.shape

    # Words: if obs is one-hot matrix (N,M,V) -> convert to indices; otherwise assume (N,M) indices
    obs = state['words']['obs']
    if obs.ndim == 3:
        # one-hot -> compute argmax along vocab dim
        word_idx = jnp.argmax(obs, axis=-1)
    else:
        word_idx = obs  # (N,M)

    # Log prob of words: gather gen[t, v] for each (n,m)
    # z_gen (N,M) -> topics for each word; word_idx (N,M) indices
    # we build indices arrays to gather vectorized
    # topic_logp(n,m) = log(gen[z_gen[n,m], word_idx[n,m]])
    log_gen = jnp.log(gen + 1e-20)  # (G0, V)
    # gather using advanced indexing: flatten and index
    flat_topic = z_gen.reshape(-1)     # (N*M,)
    flat_word  = word_idx.reshape(-1)
    flat_logp = log_gen[flat_topic, flat_word]   # (N*M,)
    word_logp_per_doc = flat_logp.reshape(N, M).sum(axis=-1)  # (N,)

    # Regression log-prob: log Normal for each doc using its assigned reg topic
    # assume sigma is std; if variance, adjust accordingly
    # compute logpdf per doc: -0.5*log(2*pi*sigma^2) - 0.5*((y-mu)^2 / sigma^2)
    reg_topic = z_reg  # (N,)
    reg_mu = mu[reg_topic]
    reg_sigma = sigma[reg_topic]
    var = reg_sigma ** 2 + 1e-12
    reg_const = -0.5 * jnp.log(2 * jnp.pi * var)
    reg_term = -0.5 * ((reg_y - reg_mu) ** 2) / var
    reg_logp_per_doc = reg_const + reg_term  # (N,)

    # sum across docs
    total_log = jnp.sum(word_logp_per_doc + reg_logp_per_doc)
    return total_log.item()  # return Python float for compatibility

# -------------------------
# Dirichlet / component generation
# -------------------------
@partial(jit, static_argnums=(0,))
def generate_distinct_components(K: int, V: int, key: jax.random.PRNGKey, peak_strength: float = 5.0, base_concent: float = 0.1):
    """
    Create K Dirichlet-distributed components of dim V, each with a strong peak at a different vocabulary location.
    Returns array (K, V).
    """
    # We sample by gamma then normalize to avoid relying on numpyro.sample API here.
    keys = random.split(key, K)
    def mk_one(k, idx):
        # alpha: base plus a peak at position idx * V // K
        alpha = jnp.ones((V,)) * base_concent
        peak_idx = (idx * V) // K
        alpha = alpha.at[peak_idx].add(peak_strength)
        g = random.gamma(k, alpha)
        return g / jnp.sum(g)
    return vmap(mk_one)(keys, jnp.arange(K))

@partial(jit, static_argnums=(0,))
def generate_distinct_distributions(N: int, V: int, key: jax.random.PRNGKey, concent: float = 0.8):
    """
    Sample N Dirichlet distributions with symmetric alpha=concent. Returns (N, V)
    """
    alpha = jnp.ones((V,)) * concent
    keys = random.split(key, N)
    def sample_dir(k):
        g = random.gamma(k, alpha)
        return g / jnp.sum(g)
    return vmap(sample_dir)(keys)

# -------------------------
# JS divergence
# -------------------------
@jit
def js_divergence(p: jnp.ndarray, q: jnp.ndarray, eps: float = 1e-12) -> jnp.ndarray:
    """
    p, q: [..., V] probability vectors
    returns JS divergence with shape [...]
    """
    p = jnp.clip(p, a_min=eps)
    q = jnp.clip(q, a_min=eps)
    m = 0.5 * (p + q)
    kl_pm = jnp.sum(p * (jnp.log(p) - jnp.log(m)), axis=-1)
    kl_qm = jnp.sum(q * (jnp.log(q) - jnp.log(m)), axis=-1)
    return 0.5 * (kl_pm + kl_qm)

# -------------------------
# select_most_diverse (greedy max-min) implemented JIT-friendly
# -------------------------
def _pairwise_distances(samples):
    # samples: (N, D)
    diff = samples[:, None, :] - samples[None, :, :]  # (N, N, D)
    return jnp.linalg.norm(diff, axis=-1)            # (N, N)

@jit
def select_most_diverse(samples: jnp.ndarray, num_select: int = 3):
    """
    Greedy max-min selection (JIT-friendly).
    samples: (N, D)
    returns: indices (num_select,)
    """
    N = samples.shape[0]
    # compute pairwise distances once
    Dmat = _pairwise_distances(samples)  # (N, N)

    # initialize selected[0] = 0
    def body_fun(state, _):
        selected, min_dists, step = state
        # mask out already selected indices by setting their min_dists to -inf
        masked = min_dists.at[selected].set(-1.0)
        # pick argmax of masked min distances
        next_idx = jnp.argmax(masked)
        # update selected list: we keep it as array of length num_select filled with -1 initially
        selected = selected.at[step].set(next_idx)
        # update min_dists: for each j, min(min_dists[j], Dmat[next_idx, j])
        new_min = jnp.minimum(min_dists, Dmat[next_idx])
        return (selected, new_min, step + 1), None

    # initial min distances: +inf except self
    init_min = jnp.full((N,), jnp.inf)
    # first selected is 0
    init_selected = -jnp.ones((num_select,), dtype=jnp.int32)
    init_selected = init_selected.at[0].set(0)
    # initial min distances updated using first selected
    init_min = jnp.minimum(init_min, Dmat[0])

    # run loop for steps 1..num_select-1
    steps = num_select - 1
    (selected_final, min_final, _), _ = lax.scan(
        lambda carry, i: (body_fun(carry, None)[0], None),
        (init_selected, init_min, 1),
        jnp.arange(steps)
    )
    # selected_final may have unfilled suffix -1 if num_select > N; clip
    selected_final = jnp.where(selected_final < 0, 0, selected_final)
    return selected_final

# -------------------------
# generate_hierarchical_mixture_data (JAX version, vectorized)
# -------------------------
@partial(jit, static_argnums=(0,1,2,3,4))
def generate_hierarchical_mixture_data(struct_upbd,
                                       N_per_base: int = 20, M: int = 200, V: int = 100,
                                       seed: int = 42):
    """
    Create synthetic hierarchical mixture data in JAX. Returns dict with:
      x: (total, M, V) one-hot word vectors (float32)
      y: (total,) regression targets
      super_labels: (total,)
      base_labels: (total,)
      word_dists: (G0, V)
      super_mix_weights: (G1, G0) or (G1, G0) style
      child_mix_weights: (G1, num_base_per_super, G0) prototypes
    NOTES:
      - To keep things JIT-friendly we avoid Python loops across samples; we vectorize generation where possible.
      - This routine returns one-hot encoded docs for direct use with your original pipeline; if you prefer indices, convert outside.
    """
    key = random.PRNGKey(seed)
    G0 = struct_upbd["G0"]
    G1 = struct_upbd["G1"]
    G2 = struct_upbd["G2"]

    total_bases = G1 * G2
    total_docs = total_bases * N_per_base

    # regression means for each mixture component index 0..G0-1
    y_means = jnp.linspace(-G0/4, G0/4, num=G0)
    # small random stds
    key, k = random.split(key)
    y_stds = 0.1 + 0.1 * random.uniform(k, (G0,))

    # create coarse super prototypes: treat them as distributions over a compact window in feature space
    assert G0 >= G2 * G1, "G0 must be >= G1*G2 for this generation scheme"
    super_window = G0 // G1
    # generate distinct small distributions for super prototypes
    key, k = random.split(key)
    super_core = generate_distinct_distributions(G1, super_window, k, concent=0.8)  # (G1, super_window)

    # embed super_core into full G0 dims by tiling blocks
    super_prototypes = jnp.zeros((G1, G0))
    # build blocks
    for i in range(G0 // super_window):
        start = i * super_window
        end = start + super_window
        # which super has this block? cycle index
        reuse = (jnp.arange(G1) % (G0 // super_window)) == i
        # broadcast super_core across blocks; this is a simple deterministic embed
        super_prototypes = super_prototypes.at[:, start:end].set(super_core * 1.0)

    # normalize
    super_prototypes = super_prototypes / jnp.sum(super_prototypes, axis=-1, keepdims=True)

    # base prototypes: for each super, pick G2 diverse bases derived from super_prototype
    base_prototypes = jnp.zeros((G1, G2, G0))
    key, k = random.split(key)
    # We'll sample candidate pool and select most diverse with the greedy selection implemented above
    for s in range(G1):
        # sample candidate pool size = G2 * 5 (juice)
        key, k = random.split(key)
        cand_pool = generate_distinct_distributions(G2 * 5, G0, k, concent=6.0 * super_prototypes[s])
        # greedily select G2 distinct prototypes indices with select_most_diverse (non-jit loop ok here)
        sel = select_most_diverse(cand_pool, num_select=G2)
        base_prototypes = base_prototypes.at[s].set(cand_pool[sel])

    # word distributions for components (G0, V)
    key, k = random.split(key)
    word_dists = generate_distinct_components(G0, V, k, peak_strength=5.0, base_concent=1.0)  # (G0, V)

    # create docs: for each base prototype, create N_per_base docs by sampling M words each
    # For vectorization: for each base (total_bases,) create N_per_base docs -> we can flatten
    # Build an array of base indices for each doc, and their super idx
    base_indices = jnp.repeat(jnp.arange(total_bases), N_per_base)  # (total_docs,)
    super_indices = base_indices // G2

    # For each doc, choose a doc-level distribution = base_prototype[super, base_in_super]
    # compute base_in_super = base_indices % G2
    base_in_super = base_indices % G2
    # gather doc mixture distributions: (total_docs, G0)
    doc_mixtures = base_prototypes[super_indices, base_in_super]  # (total_docs, G0)

    # sample M component indices per doc using vectorized categorical (Gumbel)
    key, k = random.split(key)
    keys_docs = random.split(k, total_docs)
    # prepare logits = log(doc_mixtures) with shape (total_docs, G0, 1) to sample M times independently
    logit_fn = lambda rng, logits: jax.random.categorical(rng, jnp.log(logits + 1e-20), shape=(M,))
    # vectorized sampling of M component indices per doc
    comp_samples = vmap(logit_fn)(keys_docs, doc_mixtures)  # (total_docs, M)

    # now sample word ids for each component via vectorized categorical on word_dists
    key, k = random.split(key)
    keys_word = random.split(k, total_docs * M)
    keys_word = keys_word.reshape(total_docs, M, 2)[:, :, 0]  # single key per sample for simplicity
    # flatten comp_samples
    flat_comp = comp_samples.reshape(-1)  # (total_docs * M,)
    flat_word_keys = keys_word.reshape(-1)
    # vectorized sample words
    def sample_word(rng, comp_idx):
        return random.categorical(rng, jnp.log(word_dists[comp_idx] + 1e-20), shape=())
    flat_words = vmap(sample_word)(flat_word_keys, flat_comp)  # (total_docs * M,)
    word_ids = flat_words.reshape(total_docs, M)  # (total_docs, M)

    # convert word_ids to one-hot (N, M, V) for compatibility with original pipeline
    one_hot = jax.nn.one_hot(word_ids, V)

    # regression targets: for each doc sample a component id (we reuse comp_samples[:, 0]) and draw y ~ N(mean, std)
    doc_comp_for_y = comp_samples[:, 0]
    key, k = random.split(key)
    keys_y = random.split(k, total_docs)
    def sample_y(rng, comp):
        mu = y_means[comp]
        sigma = y_stds[comp]
        return random.normal(rng) * sigma + mu
    ys = vmap(sample_y)(keys_y, doc_comp_for_y)

    return {
        "x": one_hot,                           # (total_docs, M, V)
        "y": ys,                                # (total_docs,)
        "super_labels": super_indices,          # (total_docs,)
        "base_labels": base_indices,            # (total_docs,)
        "word_dists": word_dists,               # (G0, V)
        "super_mix_weights": super_prototypes,  # (G1, G0)
        "child_mix_weights": base_prototypes    # (G1, G2, G0)
    }
